# Étape 4 — Manipulations : SPARQL avancé

On part d'une requête simple et on ajoute, cellule après cellule : `FILTER`, puis `OPTIONAL`, puis `BIND`, puis les fonctions de chaînes et de dates.

> Prérequis : `pip install -r requirements.txt`. Exécute les cellules dans l'ordre.

In [ ]:
from rdflib import Graph

graphe = Graph()
graphe.parse("personnes.ttl", format="turtle")
print(f"{len(graphe)} triplets charges.\n")


def afficher(requete: str) -> None:
    """Execute une requete SPARQL et affiche chaque ligne de resultat."""
    resultats = graphe.query(requete)
    if not resultats:
        print("  (aucun resultat)")
        return
    for ligne in resultats:
        valeurs = {str(var): str(val) for var, val in ligne.asdict().items()}
        print(" ", valeurs)

34 triplets charges.



## 1. FILTER avec un ET (`&&`)

On veut les personnes dont l'âge est **entre 30 et 40 ans**. On combine deux conditions avec `&&`.

In [ ]:
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom ?age
WHERE {
    ?p foaf:name ?nom .
    ?p foaf:age ?age .
    FILTER (?age >= 30 && ?age < 40)
}
""")

  {'nom': 'Alice Martin', 'age': '32'}
  {'nom': 'Anna Bianchi', 'age': '35'}


## 2. Le problème que résout OPTIONAL

D'abord, observons le piège. On demande nom **et** email. Comme Carol et Anna n'ont pas d'email, elles **disparaissent** complètement du résultat.

In [ ]:
# Email OBLIGATOIRE -> Carol et Anna sont exclues
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom ?email
WHERE {
    ?p foaf:name ?nom .
    ?p foaf:mbox ?email .
}
""")

  {'nom': 'Alice Martin', 'email': 'mailto:alice@exemple.org'}
  {'nom': 'Bob Dupont', 'email': 'mailto:bob@exemple.org'}
  {'nom': 'David Leroy', 'email': 'mailto:david@exemple.org'}


Maintenant avec `OPTIONAL` : tout le monde apparaît, et l'email est simplement **vide** quand il n'existe pas.

In [ ]:
# Email OPTIONNEL -> tout le monde apparait
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom ?email
WHERE {
    ?p foaf:name ?nom .
    OPTIONAL { ?p foaf:mbox ?email . }
}
""")

  {'nom': 'Alice Martin', 'email': 'mailto:alice@exemple.org'}
  {'nom': 'Bob Dupont', 'email': 'mailto:bob@exemple.org'}
  {'nom': 'Carol Nguyen'}
  {'nom': 'David Leroy', 'email': 'mailto:david@exemple.org'}
  {'nom': 'Anna Bianchi'}


## 3. Trouver ce qui MANQUE : `!BOUND`

Couplé à `OPTIONAL`, le test `!BOUND(?email)` garde uniquement les lignes où l'email est resté vide → les personnes **sans** email.

In [ ]:
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom
WHERE {
    ?p foaf:name ?nom .
    OPTIONAL { ?p foaf:mbox ?email . }
    FILTER (!BOUND(?email))
}
""")

  {'nom': 'Carol Nguyen'}
  {'nom': 'Anna Bianchi'}


## 4. Fonctions de chaînes : `STRSTARTS`

On garde les personnes dont le nom commence par "A".

In [ ]:
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom
WHERE {
    ?p foaf:name ?nom .
    FILTER (STRSTARTS(?nom, "A"))
}
""")

  {'nom': 'Alice Martin'}
  {'nom': 'Anna Bianchi'}


## 5. Fonctions de dates + `BIND` : calculer l'âge approximatif

On extrait l'année de naissance avec `YEAR(...)`, puis on calcule un âge approximatif avec `BIND`. On trie par âge décroissant avec `ORDER BY`.

In [ ]:
afficher("""
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX schema: <https://schema.org/>

SELECT ?nom ?annee ?ageApprox
WHERE {
    ?p foaf:name ?nom .
    ?p schema:birthDate ?naissance .
    BIND (YEAR(?naissance) AS ?annee)
    BIND (2026 - YEAR(?naissance) AS ?ageApprox)
}
ORDER BY DESC(?ageApprox)
""")

  {'nom': 'Carol Nguyen', 'annee': '1985', 'ageApprox': '41'}
  {'nom': 'Anna Bianchi', 'annee': '1991', 'ageApprox': '35'}
  {'nom': 'Alice Martin', 'annee': '1994', 'ageApprox': '32'}
  {'nom': 'David Leroy', 'annee': '1997', 'ageApprox': '29'}
  {'nom': 'Bob Dupont', 'annee': '1999', 'ageApprox': '27'}


## 6. À toi de jouer : 'A' OU sans email

**Exercice :** trouve les personnes dont le nom commence par 'A' **OU** qui n'ont pas d'email. Affiche nom et email (vide si absent).

Complète la cellule (combine ce que tu as vu en 3 et 4).

In [ ]:
# TODO : remplace CONDITION par ta condition, puis enleve le # de la ligne afficher(...)
ma_requete = """
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom ?email
WHERE {
    ?p foaf:name ?nom .
    OPTIONAL { ?p foaf:mbox ?email . }
    FILTER ( CONDITION )
}
"""
# afficher(ma_requete)
print("Remplace CONDITION, puis decommente la ligne afficher(ma_requete).")

Remplace CONDITION, puis decommente la ligne afficher(ma_requete).


<details>
<summary>Afficher la solution</summary>

```python
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom ?email
WHERE {
    ?p foaf:name ?nom .
    OPTIONAL { ?p foaf:mbox ?email . }
    FILTER ( STRSTARTS(?nom, "A") || !BOUND(?email) )
}
""")
```

Résultat : Alice (avec email), Anna (sans), Carol (sans).

</details>

---

Tu maîtrises l'interrogation. À l'**étape 5**, on change de sujet : décrire le **schéma** des données avec RDFS (classes, sous-classes, domain/range).